In [1]:
import os
os.chdir("..")
print(os.getcwd())

c:\Users\NASA2004\OneDrive\Documents\transformer-from-scratch


In [ ]:
# producing bleu score

import re
import torch
from tokenizers import Tokenizer
import sacrebleu

from src.model import Transformer
from src.evaluate import translate_sentence

# config
D_MODEL = 256
NUM_HEADS = 8
D_FF = 1024
NUM_LAYERS = 3
MAX_LEN = 100
DROPOUT = 0.1

tokenizer = Tokenizer.from_file("data/tokenizer/multi30k_bpe.json")
PAD_ID = tokenizer.token_to_id("<pad>")
VOCAB_SIZE = tokenizer.get_vocab_size()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_model(checkpoint_path):
    model = Transformer(VOCAB_SIZE, D_MODEL, NUM_HEADS, D_FF, NUM_LAYERS, MAX_LEN, DROPOUT, pad_id=PAD_ID).to(device)
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model.eval()
    return model

def evaluate_bleu(model, src_path, ref_path, max_len=MAX_LEN):
    with open(src_path, encoding="utf-8") as f:
        src_sentences = [line.strip() for line in f]
    with open(ref_path, encoding="utf-8") as f:
        ref_sentences = [line.strip() for line in f]

    hypotheses = []
    for sentence in src_sentences:
        translation = translate_sentence(model, sentence, tokenizer, device, max_len=max_len)
        translation = translation.replace("<bos>", "").replace("<eos>", "").strip()
        hypotheses.append(translation)

    bleu = sacrebleu.corpus_bleu(hypotheses, [ref_sentences])
    return bleu, hypotheses

# evaluating epoch 10 (best val loss) and epoch 20 (final)
for epoch in [10, 20]:
    model = load_model(f"checkpoints/epoch_{epoch}.pt")
    bleu, hyps = evaluate_bleu(model, "data/raw/test.en", "data/raw/test.de")
    print(f"Epoch {epoch}: BLEU = {bleu.score:.2f}")
    print(f"  Sample: {hyps[0]}")

That's 100 lines that end in a tokenized period ('.')
It looks like you forgot to detokenize your test data, which may hurt your score.
If you insist your data is detokenized, or don't care, you can suppress this message with the `force` parameter.


Epoch 10: BLEU = 22.72
  Sample: Ein Mann mit einem orangefarbenen Hut starrt auf etwas .


That's 100 lines that end in a tokenized period ('.')
It looks like you forgot to detokenize your test data, which may hurt your score.
If you insist your data is detokenized, or don't care, you can suppress this message with the `force` parameter.


Epoch 20: BLEU = 22.18
  Sample: Ein Mann mit einem orangefarbenen Hut starrt auf etwas .


In [3]:
# applying tokenized text for better blew scores

import re
import torch
from tokenizers import Tokenizer
import sacrebleu

from src.model import Transformer
from src.evaluate import translate_sentence

# config
D_MODEL = 256
NUM_HEADS = 8
D_FF = 1024
NUM_LAYERS = 3
MAX_LEN = 100
DROPOUT = 0.1

tokenizer = Tokenizer.from_file("data/tokenizer/multi30k_bpe.json")
PAD_ID = tokenizer.token_to_id("<pad>")
VOCAB_SIZE = tokenizer.get_vocab_size()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_model(checkpoint_path):
    model = Transformer(VOCAB_SIZE, D_MODEL, NUM_HEADS, D_FF, NUM_LAYERS, MAX_LEN, DROPOUT, pad_id=PAD_ID).to(device)
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model.eval()
    return model

def detokenize(text):
    text = re.sub(r'\s+([.,!?;:])', r'\1', text)  # removing space before punctuation
    return text.strip()

def evaluate_bleu(model, src_path, ref_path, max_len=MAX_LEN):
    with open(src_path, encoding="utf-8") as f:
        src_sentences = [line.strip() for line in f]
    with open(ref_path, encoding="utf-8") as f:
        ref_sentences = [line.strip() for line in f]

    hypotheses = []
    for sentence in src_sentences:
        translation = translate_sentence(model, sentence, tokenizer, device, max_len=max_len)
        translation = translation.replace("<bos>", "").replace("<eos>", "").strip()
        translation = detokenize(translation)
        hypotheses.append(translation)

    ref_sentences_clean = [detokenize(r) for r in ref_sentences]

    bleu = sacrebleu.corpus_bleu(hypotheses, [ref_sentences_clean])
    return bleu, hypotheses

# evaluating epoch 10 (best val loss) and epoch 20 (final)
for epoch in [10, 20]:
    model = load_model(f"checkpoints/epoch_{epoch}.pt")
    bleu, hyps = evaluate_bleu(model, "data/raw/test.en", "data/raw/test.de")
    print(f"Epoch {epoch}: BLEU = {bleu.score:.2f}")
    print(f"  Sample: {hyps[0]}")

Epoch 10: BLEU = 22.72
  Sample: Ein Mann mit einem orangefarbenen Hut starrt auf etwas.
Epoch 20: BLEU = 22.18
  Sample: Ein Mann mit einem orangefarbenen Hut starrt auf etwas.
